# Recurrent-Depth Transformer: Scaling Test-Time Compute through Latent Iteration

> *A from-scratch walkthrough of the architecture behind **Huginn-3.5B*** (Geiping et al., 2025) and its open-source sibling **[OpenMythos](https://github.com/kyegomez/OpenMythos)**.

**What you will build.** A transformer whose *depth is not fixed*. Instead, a small "core" block is iterated an *arbitrary number of times* at inference, letting the model spend more FLOPs on harder inputs **without emitting any chain-of-thought tokens**. Reasoning happens entirely in latent space.

**What is *not* re-derived here.** Multi-head attention, RMSNorm, and standard transformer FFNs are used as building blocks with minimal commentary — you already know them. Every line concerning recurrence, injection, truncated BPTT, latent-state initialisation, and adaptive compute is implemented *from scratch*.

**Outline**

| § | Topic |
|---|---|
| 1 | Motivation: why put recurrence in *depth* instead of *width* or *token-space* |
| 2 | Transformer building blocks (reused, not re-explained) |
| 3 | The three-part architecture: **Prelude → Recurrent Core → Coda** |
| 4 | Sampling recurrence counts: the Poisson log-normal schedule |
| 5 | Truncated backpropagation through depth |
| 6 | A synthetic task that *requires* variable depth: variable-length parity |
| 7 | Training loop |
| 8 | Test-time compute scaling: accuracy vs. number of iterations |
| 9 | Adaptive computation via latent-state convergence (early exit) |
| 10 | Visualising latent trajectories |


---
# 1) Motivation and intuition

## 1.1 The problem with fixed depth

A standard transformer with $L$ layers allocates the *same* amount of compute to every input. But a prime-factorisation prompt clearly needs more thought than "what colour is the sky?". There are two established ways to let a model spend more compute on hard inputs:

1. **Chain-of-thought (CoT).** Emit reasoning tokens. Compute scales with sequence length. Costs a *long*, *explicit* monologue and a large KV cache.
2. **Mixture-of-depths / early exit.** Skip or exit layers adaptively. Bounded above by the static depth $L$.

## 1.2 A third option: recurrence in depth

**Recurrent-Depth transformers** interpose a small transformer block $f_\theta$ that is applied to a *latent state* $\mathbf{s}_t$ many times:

$$
\mathbf{s}_{t+1} \;=\; f_\theta\!\big(\,[\mathbf{s}_t;\,\mathbf{e}],\,\theta\,\big),\qquad t = 0,1,\dots,R-1
$$

where $\mathbf{e}$ is the embedded input (re-injected at every step). The number of iterations $R$ is **chosen at inference time**. The model can reason silently for as many steps as compute allows, with no extra tokens and no extra parameters.

## 1.3 Relation to prior work

| Idea | Shared weights | Latent iteration | Variable depth at test time |
|---|---|---|---|
| Universal Transformer (Dehghani '19) | ✓ | ✓ | partial (ACT) |
| ALBERT (Lan '20) | ✓ | ✗ | ✗ |
| Deep Equilibrium (Bai '19) | ✓ | ✓ (fixed-point) | ✓ |
| **Huginn / OpenMythos** | ✓ | ✓ | **✓ (trained for it)** |

The key novelty of Huginn is not the architecture *per se* — it is the **training recipe** that *teaches* the model to remain useful across a broad range of iteration counts. That recipe (randomised depth + truncated BPTT) is the heart of this notebook.


In [ ]:
# Standard imports. Nothing exotic — PyTorch + matplotlib are sufficient.
import math
import random
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility (we still sample recurrence counts stochastically during training).
SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

---
# 2) Transformer building blocks

We briefly set up RMSNorm, causal multi-head attention, and a pre-norm transformer block. These are *not* the focus — they are the substrate on which the recurrent core runs.

### Sample I/O (shape-wise)
- Input: `(batch_num, seq_len, model_dim)`
- Output: `(batch_num, seq_len, model_dim)`

A single `TransformerBlock` is a drop-in Pre-Norm residual layer; we will later compose three groups of them (*prelude / core / coda*).


In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm — we implement it rather than use nn.RMSNorm for portability
    (nn.RMSNorm exists only in recent PyTorch versions)."""

    def __init__(self, model_dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(model_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


class CausalMultiHeadAttention(nn.Module):
    """Standard causal self-attention, implemented with PyTorch's fused
    scaled_dot_product_attention for speed. Not the focus of this notebook."""

    def __init__(self, model_dim: int, num_heads: int):
        super().__init__()
        assert model_dim % num_heads == 0, "model_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads
        self.qkv = nn.Linear(model_dim, 3 * model_dim, bias=False)
        self.out = nn.Linear(model_dim, model_dim, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        batch_num, seq_len, model_dim = x.shape

        # Project once, then split into Q, K, V
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, 3 * model_dim)
        qkv = self.qkv(x)

        # Reshape each of Q, K, V into heads
        # (batch_num, seq_len, model_dim) → (batch_num, num_heads, seq_len, head_dim)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_num, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Fused causal attention
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, num_heads, seq_len, head_dim)
        attn = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        # Merge heads back
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, model_dim)
        attn = attn.transpose(1, 2).contiguous().view(batch_num, seq_len, model_dim)
        return self.out(attn)


class TransformerBlock(nn.Module):
    """Pre-norm transformer block. Used identically in prelude, core, and coda."""

    def __init__(self, model_dim: int, num_heads: int, ff_mult: int = 4):
        super().__init__()
        self.norm1 = RMSNorm(model_dim)
        self.attn = CausalMultiHeadAttention(model_dim, num_heads)
        self.norm2 = RMSNorm(model_dim)
        self.mlp = nn.Sequential(
            nn.Linear(model_dim, ff_mult * model_dim),
            nn.GELU(),
            nn.Linear(ff_mult * model_dim, model_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

---
# 3) The Recurrent-Depth architecture

## 3.1 Three-stage structure

The model is partitioned into **three disjoint parameter groups**:

```
                      ┌──────────────┐
    input_ids ──► E ──► Prelude  P   ├──► e  ──────────────┐  (kept fixed, re-injected every step)
                      └──────────────┘                     │
                                                           │
                      ┌──────────────┐    s_t      [s_t ; e]
   s_0 ~ N(0,σ²) ───► │  Adapter A   │◄────────────────────┘
                      │      +        │
                      │  Core  C      │──► s_{t+1}
                      └──────────────┘           │
                              ▲                  │
                              └── iterate R times ┘
                                                           │
                                                           ▼
                                                    ┌──────────────┐
                                                    │   Coda  K    │──► logits
                                                    └──────────────┘
```

- **Prelude $P$** (a few layers): bulk, non-recurrent feature extraction.
- **Recurrent Core $C$** (one or two layers, weight-shared across iterations): where *all* variable compute happens.
- **Coda $K$** (a few layers): maps the converged latent back to output logits.

## 3.2 Why split into three?

You could imagine a "pure" recurrent model — every layer iterated. Empirically this is unstable because (i) the embedding and output surfaces need non-recurrent capacity, and (ii) gradients through many iterations of *many* layers become hopeless. The prelude/coda absorb the easy non-recurrent work; the core is kept small so one iteration is cheap.

## 3.3 The injection trick

At every recurrent step we concatenate the current latent state $\mathbf{s}_t$ with the *original* embedded input $\mathbf{e}$ and project back down:

$$
\mathbf{s}_{t+1} \;=\; C\!\big(A([\mathbf{s}_t;\mathbf{e}])\big).
$$

Without this re-injection the core would have to memorise the input in $\mathbf{s}_t$ — defeating the point. With it, the core can treat $\mathbf{e}$ as a *fixed prompt* and devote $\mathbf{s}_t$ to *reasoning state*.

## 3.4 Initial state $\mathbf{s}_0$

Sampled as Gaussian noise, $\mathbf{s}_0 \sim \mathcal{N}(0,\,\tfrac{2}{d})$. Randomness here is crucial:
- It prevents the core from collapsing to an identity map.
- It creates an implicit ensemble across seeds at inference.
- It experimentally encourages the *fixed-point* dynamics observed in Huginn's trajectories.


In [ ]:
class RecurrentDepthTransformer(nn.Module):
    """
    Three-stage recurrent-depth transformer (Huginn-style).

    Parameters
    ----------
    vocab_size           : size of the input/output token vocabulary
    model_dim            : hidden size (shared by prelude, core, and coda)
    num_heads            : attention heads per block
    num_prelude_layers   : depth of the non-recurrent prelude (P)
    num_core_layers      : depth of the recurrent core (C); applied repeatedly
    num_coda_layers      : depth of the non-recurrent coda  (K)
    max_len              : maximum supported seq_len (for positional embeddings)
    """

    def __init__(
        self,
        vocab_size: int,
        model_dim: int,
        num_heads: int,
        num_prelude_layers: int,
        num_core_layers: int,
        num_coda_layers: int,
        max_len: int,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.max_len = max_len

        # Token and positional embeddings (learned, for simplicity).
        self.tok_embed = nn.Embedding(vocab_size, model_dim)
        self.pos_embed = nn.Embedding(max_len, model_dim)

        # Prelude: frontloads non-recurrent capacity. Runs once per forward.
        self.prelude = nn.ModuleList(
            [TransformerBlock(model_dim, num_heads) for _ in range(num_prelude_layers)]
        )

        # Adapter A: fuses previous latent state s_t with the (frozen) embedding e.
        # Concatenating along the feature axis then projecting back to model_dim
        # is the simplest learnable "injection" operator — equivalent to a gated
        # sum but cheaper.
        # (batch_num, seq_len, 2 * model_dim) → (batch_num, seq_len, model_dim)
        self.adapter = nn.Linear(2 * model_dim, model_dim, bias=False)

        # Recurrent core: these weights are shared across ALL iterations. The
        # same nn.ModuleList is walked every step.
        self.core = nn.ModuleList(
            [TransformerBlock(model_dim, num_heads) for _ in range(num_core_layers)]
        )

        # Coda: decodes the final latent back to vocab logits.
        self.coda = nn.ModuleList(
            [TransformerBlock(model_dim, num_heads) for _ in range(num_coda_layers)]
        )

        self.final_norm = RMSNorm(model_dim)
        self.head = nn.Linear(model_dim, vocab_size, bias=False)

    # ------------------------------------------------------------------
    # Modular forward primitives
    # ------------------------------------------------------------------
    def embed(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Token embed + positional embed + prelude."""
        # (batch_num, seq_len) → (batch_num, seq_len, model_dim)
        batch_num, seq_len = input_ids.shape
        assert seq_len <= self.max_len, "seq_len exceeds max_len"

        # Position ids broadcast across the batch
        pos = torch.arange(seq_len, device=input_ids.device)
        x = self.tok_embed(input_ids) + self.pos_embed(pos)

        for block in self.prelude:
            x = block(x)
        return x

    def iterate(self, s: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
        """A single recurrent step: s_{t+1} = Core(Adapter([s_t; e]))."""
        # Concatenate along the feature axis, then project back down
        # (batch_num, seq_len, 2 * model_dim) → (batch_num, seq_len, model_dim)
        x = self.adapter(torch.cat([s, e], dim=-1))
        for block in self.core:
            x = block(x)
        return x

    def decode(self, s: torch.Tensor) -> torch.Tensor:
        """Coda + norm + unembedding head."""
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, vocab_size)
        for block in self.coda:
            s = block(s)
        return self.head(self.final_norm(s))

    def init_state(self, e: torch.Tensor) -> torch.Tensor:
        """Random Gaussian latent state with sigma = sqrt(2 / model_dim).

        This scale matches Kaiming init for a 'zero input' — it keeps the
        first iteration from saturating attention softmaxes."""
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        sigma = math.sqrt(2.0 / self.model_dim)
        return torch.randn_like(e) * sigma

    # ------------------------------------------------------------------
    # Full forward with truncated BPTT split
    # ------------------------------------------------------------------
    def forward(
        self,
        input_ids: torch.Tensor,
        num_recur_steps: int,
        num_backprop_steps: Optional[int] = None,
        return_trajectory: bool = False,
    ):
        """Run the model with `num_recur_steps` total iterations, of which
        only the *last* `num_backprop_steps` are differentiable.

        This is the truncated BPTT split explained in §5."""
        # (batch_num, seq_len) → (batch_num, seq_len, model_dim)
        e = self.embed(input_ids)

        # (batch_num, seq_len, model_dim)
        s = self.init_state(e)

        # Default: differentiate through every step (dense mode)
        if num_backprop_steps is None:
            num_backprop_steps = num_recur_steps
        num_backprop_steps = min(num_backprop_steps, num_recur_steps)
        num_nograd = num_recur_steps - num_backprop_steps

        trajectory = [] if return_trajectory else None

        # Warm-up under no_grad: constant memory regardless of num_nograd
        with torch.no_grad():
            for _ in range(num_nograd):
                s = self.iterate(s, e)
                if trajectory is not None:
                    trajectory.append(s.detach().clone())

        # Explicitly sever the graph — the embedding e remains attached, but
        # the iterated state does not.
        s = s.detach()

        # Gradient window: only these iterations contribute to ∂L/∂θ_core.
        for _ in range(num_backprop_steps):
            s = self.iterate(s, e)
            if trajectory is not None:
                trajectory.append(s.detach().clone())

        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, vocab_size)
        logits = self.decode(s)
        if return_trajectory:
            return logits, s, trajectory
        return logits, s

### Shape sanity check

Before going further we verify the forward pass composes correctly at different recurrence counts.


In [ ]:
# Instantiate a tiny model on CPU for the shape check
toy_model = RecurrentDepthTransformer(
    vocab_size=2,
    model_dim=32,
    num_heads=4,
    num_prelude_layers=1,
    num_core_layers=1,
    num_coda_layers=1,
    max_len=16,
).to(DEVICE)

# (batch_num, seq_len)
dummy = torch.randint(0, 2, (4, 16), device=DEVICE)

# Verify output shape at several iteration counts
for r in [1, 4, 16]:
    logits, state = toy_model(dummy, num_recur_steps=r, num_backprop_steps=min(r, 2))
    print(
        f"R = {r:>2d}:  logits shape = {tuple(logits.shape)}  "
        f"|  final state shape = {tuple(state.shape)}"
    )

# Count parameters — note that num_recur_steps does NOT increase parameter count.
num_params = sum(p.numel() for p in toy_model.parameters())
print(f"\nparameter count (independent of R): {num_params:,}")

---
# 4) Sampling recurrence counts during training

## 4.1 Why not just train with fixed $R$?

If we train with $R = 16$ only, the model *overfits* that single depth. At test time $R=4$ is underwhelming and $R=64$ drifts. We want an architecture that is *monotonically better* with more compute and useful even at very small $R$. That requires exposing it to a **wide distribution** over $R$ during training.

## 4.2 The Poisson log-normal schedule

Geiping et al. use, for each training step:

$$
\mu \sim \mathrm{LogNormal}(\log \bar\mu,\; \sigma_{\log}^2), \qquad R \sim \mathrm{Poisson}(\mu).
$$

This is a *heavy-tailed* distribution:
- The median $R$ is around $\bar\mu$ — the "typical" budget.
- Occasional very large samples teach the model to remain stable at depths well beyond typical.
- Occasional very small samples force the coda to work even with barely-iterated states.

### Sample I/O
- Input: hyperparameters `mean_log`, `std_log`, `min_n`, `max_n`.
- Output: one positive integer $R$.


In [ ]:
def sample_recurrence_count(
    mean_log: float = math.log(16.0),
    std_log: float = 0.5,
    min_n: int = 1,
    max_n: int = 64,
) -> int:
    """Draw R ~ Poisson( LogNormal(mean_log, std_log) ).

    With defaults ( mean_log = ln 16, std_log = 0.5 ) the distribution has:
        E[R] ≈ 18,   median ≈ 16,   P(R > 32) ≈ 0.17.
    """
    mu = math.exp(random.gauss(mean_log, std_log))
    # torch.poisson accepts rate directly — we cast via a scalar tensor
    n = int(torch.poisson(torch.tensor(float(mu))).item())
    return max(min_n, min(n, max_n))


# Visualise 20k samples from the schedule
samples = [sample_recurrence_count() for _ in range(20_000)]

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.hist(samples, bins=np.arange(0, 65) - 0.5, edgecolor="black", linewidth=0.3)
ax.set_xlabel("R (number of recurrent iterations)")
ax.set_ylabel("count over 20k training draws")
ax.set_title("Poisson log-normal distribution used during training")
ax.axvline(
    np.mean(samples),
    color="red",
    linestyle="--",
    label=f"mean = {np.mean(samples):.1f}",
)
ax.axvline(
    np.median(samples),
    color="orange",
    linestyle="--",
    label=f"median = {np.median(samples):.0f}",
)
ax.legend()
plt.tight_layout()
plt.show()

---
# 5) Truncated backpropagation through depth

## 5.1 The memory problem

A forward pass with $R=32$ iterations and a core of 2 transformer layers builds a computation graph **64 layers deep**. Backprop through all of it is both memory-hungry and gradient-unstable (explosion / vanishing).

## 5.2 The fix: backprop only through the last $k$ iterations

Split the $R$ iterations into two phases:

$$
\underbrace{s_0 \to s_1 \to \dots \to s_{R-k}}_{\text{no\_grad (warm-up)}} \;\;\to\;\; \underbrace{s_{R-k} \to \dots \to s_R}_{\text{autograd (gradient window)}}
$$

- The warm-up runs under `torch.no_grad()` — **constant** activation memory.
- We `.detach()` the state to sever the graph, then run $k$ more iterations *with* gradients.
- The gradient window size $k$ is usually small and fixed (e.g. $k=8$), while $R$ can be huge.

## 5.3 What does the model *learn* from this?

Because the "differentiable" starting state at iteration $R-k$ is *the output of the model's own past warm-up*, the gradient trains the core on **states it realistically sees at depth**. This is closer to how we actually deploy it (long warm-up, short interactive tail) than dense BPTT would be.

## 5.4 Sample I/O
- Input: a batch of token ids, a model, an optimiser, recurrence hyperparams.
- Output: a scalar loss after one update step, and the number of iterations used.


In [ ]:
def training_step(
    model: RecurrentDepthTransformer,
    optimizer: torch.optim.Optimizer,
    input_ids: torch.Tensor,
    targets: torch.Tensor,
    max_backprop: int = 8,
) -> tuple[float, int]:
    """One SGD step with a random recurrence depth and truncated BPTT."""
    # Draw this step's recurrence budget from the Poisson log-normal schedule
    num_recur = sample_recurrence_count()
    num_bp = min(max_backprop, num_recur)

    # Forward (warm-up is under no_grad inside the model)
    # (batch_num, seq_len) → (batch_num, seq_len, vocab_size)
    logits, _ = model(input_ids, num_recur_steps=num_recur, num_backprop_steps=num_bp)

    # Cross-entropy over every token position
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping — essential because long warm-ups can produce very
    # large state norms before the differentiable window begins.
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return loss.item(), num_recur

---
# 6) A task that benefits from variable depth: **variable-length parity**

## 6.1 Why parity?

A well-known theoretical result (Hahn 2020; Merrill & Sabharwal 2023) says that **no** transformer of constant depth can compute the parity of an arbitrary-length binary string. The function requires a chain of $\Theta(\log n)$ attention compositions, so longer inputs *provably* require more depth.

This makes parity the ideal stress-test for a *variable-depth* model: we can show that

1. At a fixed training length, more iterations monotonically improve accuracy.
2. At test time we can extrapolate to *longer* sequences by simply running more iterations — something a fixed-depth transformer cannot do.

## 6.2 Formal statement

Given a binary sequence $x_1, \dots, x_L \in \{0,1\}^L$, at each position $i$ predict

$$
y_i \;=\; x_1 \oplus x_2 \oplus \dots \oplus x_i \;\in\;\{0,1\}.
$$

### Sample I/O

```
input   : [1, 0, 1, 1, 0, 1]
target  : [1, 1, 0, 1, 1, 0]   ← cumulative XOR
```

Both inputs and targets live in a 2-symbol vocabulary, so `vocab_size = 2`.


In [ ]:
def make_parity_batch(
    batch_num: int,
    seq_len: int,
    device: str = "cpu",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Random binary sequences and their running XOR."""
    # (batch_num, seq_len) of {0, 1}
    bits = torch.randint(0, 2, (batch_num, seq_len), device=device)

    # Running XOR = cumulative sum mod 2
    # (batch_num, seq_len) → (batch_num, seq_len)
    targets = torch.cumsum(bits, dim=1) % 2
    return bits, targets


# Inspect a handful of examples
bits, targets = make_parity_batch(batch_num=3, seq_len=10)
for i in range(3):
    print(f"bits    : {bits[i].tolist()}")
    print(f"targets : {targets[i].tolist()}\n")

---
# 7) Training loop

## 7.1 Model configuration

We deliberately keep the prelude and coda shallow (1 layer each) and the core tiny (1 layer). With so few "free" non-recurrent layers, the model *must* rely on recurrence for any accuracy beyond trivial. This makes test-time compute scaling visible on a laptop.

| hyperparameter | value | rationale |
|---|---|---|
| `model_dim` | 64 | small but sufficient for a 2-symbol vocab |
| `num_heads` | 4 | 16-dim heads — fine |
| `num_prelude_layers` | 1 | minimal input processing |
| `num_core_layers` | 1 | tiny cost per iteration |
| `num_coda_layers` | 1 | minimal decoder |
| `max_len` | 128 | we'll test on lengths up to 96 |
| `max_backprop` | 8 | gradient window size $k$ |


In [ ]:
def build_model() -> RecurrentDepthTransformer:
    return RecurrentDepthTransformer(
        vocab_size=2,
        model_dim=64,
        num_heads=4,
        num_prelude_layers=1,
        num_core_layers=1,
        num_coda_layers=1,
        max_len=128,
    ).to(DEVICE)


def train(
    model: RecurrentDepthTransformer,
    num_iters: int = 1500,
    batch_num: int = 64,
    seq_len: int = 24,
    max_backprop: int = 8,
    lr: float = 3e-4,
    log_every: int = 100,
) -> list[float]:
    """Full training loop. Returns the per-step loss trace."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    losses: list[float] = []
    recent: list[float] = []

    for step in range(num_iters):
        # Fresh random batch every step — the task has effectively infinite data
        bits, targets = make_parity_batch(batch_num, seq_len, DEVICE)

        loss, num_recur = training_step(
            model, optimizer, bits, targets, max_backprop=max_backprop
        )
        losses.append(loss)
        recent.append(loss)

        if step % log_every == 0 or step == num_iters - 1:
            avg = sum(recent) / len(recent)
            recent = []
            print(
                f"step {step:5d}  |  R = {num_recur:3d}  |  "
                f"loss {loss:.4f}  |  avg(last {log_every}) {avg:.4f}"
            )
    return losses


# Reset seeds so the run is reproducible
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

model = build_model()
loss_trace = train(model, num_iters=1500)

In [ ]:
# Plot the training loss — log scale helps see late-stage improvement
fig, ax = plt.subplots(figsize=(7, 3.5))
window = 50
smoothed = np.convolve(loss_trace, np.ones(window) / window, mode="valid")
ax.plot(loss_trace, alpha=0.25, label="per-step loss")
ax.plot(
    np.arange(len(smoothed)) + window // 2,
    smoothed,
    color="red",
    label=f"rolling mean (window {window})",
)
ax.set_xlabel("training step")
ax.set_ylabel("cross-entropy loss")
ax.set_yscale("log")
ax.set_title("Training curve — loss over steps")
ax.legend()
plt.tight_layout()
plt.show()

---
# 8) Test-time compute scaling

This is the **defining experiment** of the architecture. We take the trained model, freeze its weights, and evaluate at a *sweep* of recurrence counts $R \in \{1, 2, 4, 8, 16, 32, 64, 128\}$.

We also repeat the sweep at a **longer** sequence length than seen in training. The hope: the model learned a general "XOR-scanning" algorithm in the core, and extrapolates to longer inputs *when given more iterations*.

### Sample I/O
- Input: a trained model and a list of `R` values.
- Output: accuracy at each `R` (one number per setting).


In [ ]:
@torch.no_grad()
def evaluate_accuracy(
    model: RecurrentDepthTransformer,
    num_recur_steps: int,
    batch_num: int = 128,
    seq_len: int = 24,
    num_batches: int = 20,
) -> float:
    """Token-level accuracy averaged over num_batches fresh batches."""
    model.eval()
    correct = 0
    total = 0
    for _ in range(num_batches):
        bits, targets = make_parity_batch(batch_num, seq_len, DEVICE)
        # (batch_num, seq_len) → (batch_num, seq_len, vocab_size)
        logits, _ = model(bits, num_recur_steps=num_recur_steps)
        # (batch_num, seq_len, vocab_size) → (batch_num, seq_len)
        pred = logits.argmax(dim=-1)
        correct += (pred == targets).sum().item()
        total += targets.numel()
    model.train()
    return correct / total


R_sweep = [1, 2, 4, 8, 16, 32, 64, 128]

# In-distribution length (same as training)
acc_in_dist = [evaluate_accuracy(model, r, seq_len=24) for r in R_sweep]

# Out-of-distribution: longer than any training sequence
acc_long = [evaluate_accuracy(model, r, seq_len=64) for r in R_sweep]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(R_sweep, acc_in_dist, "o-", label="seq_len = 24 (train length)")
ax.plot(R_sweep, acc_long, "s-", label="seq_len = 64 (test-time extrapolation)")
ax.set_xscale("log", base=2)
ax.set_xticks(R_sweep)
ax.set_xticklabels([str(r) for r in R_sweep])
ax.axhline(0.5, color="grey", linestyle=":", label="chance")
ax.set_xlabel("num_recur_steps R (at test time)")
ax.set_ylabel("token accuracy")
ax.set_title("Test-time compute scaling: more iterations → higher accuracy")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for r, a1, a2 in zip(R_sweep, acc_in_dist, acc_long):
    print(f"R={r:>4d}  |  acc@24 = {a1:.3f}  |  acc@64 = {a2:.3f}")

### Interpretation

- **In-distribution** (`seq_len = 24`): accuracy saturates quickly — the task is easy with moderate depth.
- **Out-of-distribution** (`seq_len = 64`): accuracy keeps climbing with $R$. The model *buys* accuracy with compute at inference.

This is the key operational claim of recurrent-depth transformers: **inference quality is a slider controlled by $R$**, not a fixed property of the weights. A fixed-depth transformer does not have this slider.


---
# 9) Adaptive computation via latent-state convergence

## 9.1 The fixed-point hypothesis

Empirically, Huginn's latent trajectories approach a *fixed point* $\mathbf{s}^\star$ such that $\mathbf{s}^\star = C(A([\mathbf{s}^\star;\mathbf{e}]))$. Once convergence sets in, further iterations change the state negligibly, so the output is stable.

## 9.2 Early-exit criterion

At inference we track the relative change:

$$
\Delta_t = \frac{\lVert \mathbf{s}_{t+1} - \mathbf{s}_t \rVert_2}{\lVert \mathbf{s}_{t+1} \rVert_2},
$$

and stop iterating as soon as $\Delta_t < \tau$ for some tolerance $\tau$. This gives **per-input adaptive compute**: easy inputs converge quickly; hard inputs use more of the budget.

### Sample I/O
- Input: `input_ids`, `max_steps`, `tol`.
- Output: `logits`, the number of steps actually used, and the $\Delta_t$ trace.


In [ ]:
@torch.no_grad()
def forward_with_early_exit(
    model: RecurrentDepthTransformer,
    input_ids: torch.Tensor,
    max_steps: int = 128,
    tol: float = 1e-3,
) -> tuple[torch.Tensor, int, list[float]]:
    """Iterate the core until the latent state stops moving, then decode."""
    model.eval()

    # (batch_num, seq_len) → (batch_num, seq_len, model_dim)
    e = model.embed(input_ids)
    s = model.init_state(e)
    deltas: list[float] = []

    for t in range(max_steps):
        s_new = model.iterate(s, e)

        # Relative change per position, then averaged over batch × seq
        num = (s_new - s).norm(dim=-1)
        den = s_new.norm(dim=-1).clamp_min(1e-8)
        delta = (num / den).mean().item()
        deltas.append(delta)

        s = s_new
        if delta < tol:
            break

    logits = model.decode(s)
    return logits, t + 1, deltas


# Compare steps used across two regimes: easy (short input) vs hard (long input)
short_in, _ = make_parity_batch(batch_num=1, seq_len=8, device=DEVICE)
long_in, _ = make_parity_batch(batch_num=1, seq_len=64, device=DEVICE)

_, steps_short, deltas_short = forward_with_early_exit(model, short_in, tol=1e-3)
_, steps_long, deltas_long = forward_with_early_exit(model, long_in, tol=1e-3)

print(f"short input (seq_len=8):  exited after {steps_short} steps")
print(f"long  input (seq_len=64): exited after {steps_long} steps")

# Plot the relative-change curves
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(deltas_short, "o-", label=f"seq_len=8  (exit @ step {steps_short})")
ax.plot(deltas_long, "s-", label=f"seq_len=64 (exit @ step {steps_long})")
ax.axhline(1e-3, color="red", linestyle="--", label="tolerance τ = 1e-3")
ax.set_yscale("log")
ax.set_xlabel("iteration t")
ax.set_ylabel(r"relative change  $\Delta_t$")
ax.set_title("Latent-state convergence drives adaptive compute")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
# 10) Visualising latent trajectories

To build intuition for what "thinking in latent space" looks like, we project the trajectory of a single position's latent state $\mathbf{s}_t^{(i)}$ for $t = 0, \dots, R$ onto its top two principal components and plot the path.

What you should see if the model has learned a useful dynamics:
- Trajectories start far from any structure (random init).
- They sweep toward a compact region of latent space.
- Similar inputs converge to similar endpoints.


In [ ]:
@torch.no_grad()
def collect_trajectory(
    model: RecurrentDepthTransformer,
    input_ids: torch.Tensor,
    num_steps: int,
    position: int = -1,
) -> torch.Tensor:
    """Return the trajectory of one chosen token position through iterations."""
    model.eval()

    # Forward collecting every intermediate state
    _, _, traj = model(
        input_ids,
        num_recur_steps=num_steps,
        num_backprop_steps=0,
        return_trajectory=True,
    )

    # traj is a list of length num_steps of (batch_num, seq_len, model_dim) tensors.
    # We extract a single (position, batch=0) vector per step.
    # (num_steps, model_dim)
    pts = torch.stack([t[0, position] for t in traj])
    return pts


def fit_pca(
    points: torch.Tensor, num_components: int = 2
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (mean, top-k principal axes) for a tensor of shape (num_samples, model_dim)."""
    # (num_samples, model_dim) → mean: (1, model_dim), axes: (num_components, model_dim)
    mean = points.mean(dim=0, keepdim=True)
    _, _, vh = torch.linalg.svd(points - mean, full_matrices=False)
    return mean, vh[:num_components]


# Collect trajectories for 8 different random inputs of the same length
num_samples = 8
num_steps = 40
all_trajs: list[torch.Tensor] = []
for _ in range(num_samples):
    bits, _ = make_parity_batch(batch_num=1, seq_len=24, device=DEVICE)
    all_trajs.append(collect_trajectory(model, bits, num_steps=num_steps))

# Fit PCA jointly on all trajectory points so the axes are shared across samples
# (num_samples * num_steps, model_dim)
joint = torch.cat(all_trajs, dim=0)
mean, proj = fit_pca(joint, num_components=2)

fig, ax = plt.subplots(figsize=(7, 5.5))
colors = plt.cm.viridis(np.linspace(0, 1, num_samples))
for idx, pts in enumerate(all_trajs):
    xy = ((pts - mean) @ proj.T).cpu().numpy()
    ax.plot(xy[:, 0], xy[:, 1], "-", color=colors[idx], alpha=0.6, linewidth=1.2)
    ax.scatter(
        xy[0, 0],
        xy[0, 1],
        color=colors[idx],
        s=40,
        marker="o",
        edgecolor="black",
        linewidth=0.6,
        label="start" if idx == 0 else None,
    )
    ax.scatter(
        xy[-1, 0],
        xy[-1, 1],
        color=colors[idx],
        s=80,
        marker="*",
        edgecolor="black",
        linewidth=0.6,
        label="end" if idx == 0 else None,
    )

ax.set_title(f"Latent trajectories over {num_steps} iterations (PCA to 2D)")
ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()